# 牛牛战法 Round35 主线跃迁启动诊断

## tl;dr

- 在查看收益前冻结两个只放宽牛牛启动分数的规则：全市场题材首次进入前五且突破，以及题材位于前五、名次继续改善且突破。
- 当前股票池两候选四窗胜率为 48.45% / 47.74%，均低于生产阶段基线 55.63%；复合收益也从 10.38% 降至 2.31% / 4.93%。
- 扩展股票池两候选为 50.49% / 49.52% 胜率、10.90% / 9.70% 收益，同样低于基线 54.49% / 14.82%。
- 两个候选冻结为 `rejected_round35`。生产评分、排序、仓位和退出不变；严格前向观察继续等待 2026-08-03 之后至少 30 笔完整交易或三个完整自然月。

## Context & Methods

本笔记本面向策略维护者，复核‘酝酿 → 主升 → 高潮 → 分歧 → 退幕’流程中主升启动覆盖不足的问题。输入是冻结的点时题材横截面宽带缓存和同一账户模型生成的回放 JSON，不访问网络或真实模拟账户。四个互斥主窗口为 `old_sealed`、`train_a`、`train_b`、`validation`；`recent` 与 validation 重叠，只作观察，不并入开发聚合。

### Key Assumptions

- 信号使用当日收盘时已经存在的题材和个股字段，成交最早发生在下一交易日开盘。
- 两个候选保留所有生产阈值信号，只对 7.0 至 8.4 分的牛牛启动增加完整横截面跃迁许可；必须为突破买点，旧窄口径排名安全拒绝。
- 账户仍使用 5bps 滑点、A 股费用、整手、现金、T+1、最多 2 个新仓/5 只持仓，以及生产退出状态机。
- 扩展股票池是代码补全敏感性上界，历史名称/ST 成员关系仍不是完整点时口径。

In [1]:
import gzip
import json
import math
import os
import statistics
from collections import Counter
from pathlib import Path

PATHS = {
    'result_current': Path(os.environ.get('NIUONE_ROUND35_CURRENT_RESULT', '/private/tmp/niuone-round35-stage-wide-current-analysis.json')),
    'result_extended': Path(os.environ.get('NIUONE_ROUND35_EXTENDED_RESULT', '/private/tmp/niuone-round35-stage-wide-extended-analysis.json')),
    'cache_current': Path(os.environ.get('NIUONE_ROUND35_CURRENT_CACHE', '/private/tmp/niuone-round34-stage-wide-full-theme-cross-section.jsonl.gz')),
    'cache_extended': Path(os.environ.get('NIUONE_ROUND35_EXTENDED_CACHE', '/private/tmp/niuone-round35-stage-wide-extended-full-theme-cross-section.jsonl.gz')),
}
for label, path in PATHS.items():
    if not path.exists():
        raise FileNotFoundError(f'{label}: {path}')
results = {
    scope: json.loads(PATHS[f'result_{scope}'].read_text(encoding='utf-8'))
    for scope in ('current', 'extended')
}
print({label: str(path) for label, path in PATHS.items()})

{'result_current': '/private/tmp/niuone-round35-stage-wide-current-analysis.json', 'result_extended': '/private/tmp/niuone-round35-stage-wide-extended-analysis.json', 'cache_current': '/private/tmp/niuone-round34-stage-wide-full-theme-cross-section.jsonl.gz', 'cache_extended': '/private/tmp/niuone-round35-stage-wide-extended-full-theme-cross-section.jsonl.gz'}


## Data

缓存检查确认交易日顺序、完整横截面覆盖、信号题材连接和研究低门槛。当前池的预收益覆盖审计还重算低分启动数量，以及两条冻结规则在四窗中的可用信号数。

In [2]:
WINDOWS = {
    'old_sealed': ('2025-05-01', '2025-07-31'),
    'train_a': ('2025-08-01', '2025-10-31'),
    'train_b': ('2025-11-01', '2026-01-31'),
    'validation': ('2026-02-01', '2026-04-30'),
}

def cache_profile(path, *, profile_rules=False):
    profile = Counter()
    previous_ranks = {}
    last_date = ''
    rule_windows = {name: Counter() for name in WINDOWS}
    with gzip.open(path, 'rt', encoding='utf-8') as handle:
        for line in handle:
            if line.startswith('{\"kind\": \"bars\"'):
                continue
            item = json.loads(line)
            if item.get('kind') != 'frame':
                continue
            date = item['date']
            profile['date_order_errors'] += int(date < last_date)
            last_date = date
            cross_section = item.get('cross_section') or {}
            scores = {
                industry: float(values['score'])
                for industry, values in cross_section.items()
                if isinstance(values, dict) and values.get('score') is not None
                and math.isfinite(float(values['score']))
            }
            ranked = sorted(scores, key=lambda industry: (-scores[industry], industry))
            ranks = {industry: rank for rank, industry in enumerate(ranked, start=1)}
            profile['frames'] += 1
            profile['cross_section_frames'] += int(bool(cross_section))
            profile['theme_rows'] += len(cross_section)
            for signal in item.get('signals') or []:
                profile['signals'] += 1
                scored = (signal.get('metadata') or {}).get('scored') or {}
                industry = str(scored.get('industry') or '').strip()
                profile['missing_signal_theme_join'] += int(bool(industry) and industry not in ranks)
                if not profile_rules or signal.get('strategy_id') != 'niu_emerging' or float(signal.get('score') or 0) >= 8.4:
                    continue
                rank = ranks.get(industry)
                previous_rank = previous_ranks.get(industry)
                new_top5 = previous_rank is not None and previous_rank > 5 and rank is not None and rank <= 5
                top5_improving = previous_rank is not None and rank is not None and rank <= 5 and rank < previous_rank
                for window_name, (start, end) in WINDOWS.items():
                    if start <= date <= end:
                        rule_windows[window_name]['low_emerging'] += 1
                        if scored.get('entry_setup') == 'breakout' and new_top5:
                            rule_windows[window_name]['new_top5_breakout'] += 1
                        if scored.get('entry_setup') == 'breakout' and top5_improving:
                            rule_windows[window_name]['top5_improving_breakout'] += 1
            previous_ranks = ranks
    return dict(profile), {name: dict(counts) for name, counts in rule_windows.items()}

for scope in ('current', 'extended'):
    profile, coverage = cache_profile(PATHS[f'cache_{scope}'], profile_rules=scope == 'current')
    print(scope, profile)
    if scope == 'current':
        print('pre-outcome rule coverage', coverage)

current {'date_order_errors': 0, 'frames': 304, 'cross_section_frames': 304, 'theme_rows': 39824, 'signals': 4468, 'missing_signal_theme_join': 0}
pre-outcome rule coverage {'old_sealed': {'low_emerging': 326, 'new_top5_breakout': 35, 'top5_improving_breakout': 39}, 'train_a': {'low_emerging': 473, 'new_top5_breakout': 49, 'top5_improving_breakout': 54}, 'train_b': {'low_emerging': 446, 'top5_improving_breakout': 40, 'new_top5_breakout': 33}, 'validation': {'low_emerging': 291, 'new_top5_breakout': 33, 'top5_improving_breakout': 39}}


extended {'date_order_errors': 0, 'frames': 304, 'cross_section_frames': 304, 'theme_rows': 39824, 'signals': 4766, 'missing_signal_theme_join': 0}


## Results

开发聚合从四个窗口逐笔重算，并与保存的汇总断言一致。随后按 `(股票, 入场日, 策略)` 比较候选与基线的共同、增加和被账户竞争挤出的成交。

In [3]:
PRIMARY_WINDOWS = tuple(WINDOWS)
CANDIDATES = (
    'stage_production_thresholds',
    'stage_emerging_score_70_full_theme_new_top5_breakout',
    'stage_emerging_score_70_full_theme_top5_improving_breakout',
)

def recompute(candidate):
    windows = candidate['windows']
    completed = wins = positive_windows = 0
    compounded = 1.0
    worst_drawdown = 0.0
    for window_name in PRIMARY_WINDOWS:
        rows = windows[window_name]['completed_trade_features']
        returns = [float(row['net_return_pct']) for row in rows]
        completed += len(returns)
        wins += sum(value > 0 for value in returns)
        portfolio = windows[window_name]['statistics']
        window_return = float(portfolio['portfolio_return_pct'])
        compounded *= 1.0 + window_return / 100.0
        positive_windows += int(window_return > 0)
        worst_drawdown = min(worst_drawdown, float(portfolio['max_drawdown_pct']))
    return {
        'completed_trade_count': completed,
        'win_count': wins,
        'win_rate_pct': round(wins / completed * 100.0, 4),
        'compounded_portfolio_return_pct': round((compounded - 1.0) * 100.0, 4),
        'positive_window_count': positive_windows,
        'worst_max_drawdown_pct': round(worst_drawdown, 4),
    }

for scope, result in results.items():
    print('\n', scope)
    for candidate_name in CANDIDATES:
        candidate = result['candidates'][candidate_name]
        checked = recompute(candidate)
        stored = candidate['development_aggregate']
        for field, value in checked.items():
            assert value == stored[field], (scope, candidate_name, field, value, stored[field])
        recent = candidate['windows']['recent']['statistics']
        print(candidate_name, {**checked, 'recent_trades': recent['completed_trade_count'], 'recent_win_rate_pct': recent['win_rate_pct'], 'recent_return_pct': recent['portfolio_return_pct']})


 current
stage_production_thresholds {'completed_trade_count': 151, 'win_count': 84, 'win_rate_pct': 55.6291, 'compounded_portfolio_return_pct': 10.3827, 'positive_window_count': 3, 'worst_max_drawdown_pct': -5.693, 'recent_trades': 16, 'recent_win_rate_pct': 68.75, 'recent_return_pct': 2.2353}
stage_emerging_score_70_full_theme_new_top5_breakout {'completed_trade_count': 194, 'win_count': 94, 'win_rate_pct': 48.4536, 'compounded_portfolio_return_pct': 2.3079, 'positive_window_count': 2, 'worst_max_drawdown_pct': -9.3963, 'recent_trades': 22, 'recent_win_rate_pct': 59.0909, 'recent_return_pct': 10.6969}
stage_emerging_score_70_full_theme_top5_improving_breakout {'completed_trade_count': 199, 'win_count': 95, 'win_rate_pct': 47.7387, 'compounded_portfolio_return_pct': 4.9301, 'positive_window_count': 3, 'worst_max_drawdown_pct': -9.9117, 'recent_trades': 23, 'recent_win_rate_pct': 60.8696, 'recent_return_pct': 9.8444}

 extended
stage_production_thresholds {'completed_trade_count': 156

In [4]:
def trade_key(row):
    return row['symbol'], row['entry_date'], row['strategy_id']

def trade_stats(rows):
    values = [float(row['net_return_pct']) for row in rows]
    return {
        'trades': len(values),
        'wins': sum(value > 0 for value in values),
        'win_rate_pct': round(sum(value > 0 for value in values) / len(values) * 100.0, 4) if values else None,
        'average_return_pct': round(sum(values) / len(values), 4) if values else None,
        'median_return_pct': round(statistics.median(values), 4) if values else None,
    }

for scope, result in results.items():
    baseline = result['candidates'][CANDIDATES[0]]['windows']
    print('\n', scope)
    for candidate_name in CANDIDATES[1:]:
        candidate_windows = result['candidates'][candidate_name]['windows']
        added = []
        displaced = []
        common = 0
        for window_name in PRIMARY_WINDOWS:
            baseline_rows = {trade_key(row): row for row in baseline[window_name]['completed_trade_features']}
            candidate_rows = {trade_key(row): row for row in candidate_windows[window_name]['completed_trade_features']}
            assert len(baseline_rows) == len(baseline[window_name]['completed_trade_features'])
            assert len(candidate_rows) == len(candidate_windows[window_name]['completed_trade_features'])
            common += len(baseline_rows.keys() & candidate_rows.keys())
            added.extend(candidate_rows[key] for key in candidate_rows.keys() - baseline_rows.keys())
            displaced.extend(baseline_rows[key] for key in baseline_rows.keys() - candidate_rows.keys())
        print(candidate_name, {'common': common, 'added': trade_stats(added), 'displaced': trade_stats(displaced)})


 current
stage_emerging_score_70_full_theme_new_top5_breakout {'common': 137, 'added': {'trades': 57, 'wins': 16, 'win_rate_pct': 28.0702, 'average_return_pct': -0.6319, 'median_return_pct': -2.2714}, 'displaced': {'trades': 14, 'wins': 6, 'win_rate_pct': 42.8571, 'average_return_pct': 2.2289, 'median_return_pct': -1.7132}}
stage_emerging_score_70_full_theme_top5_improving_breakout {'common': 135, 'added': {'trades': 64, 'wins': 19, 'win_rate_pct': 29.6875, 'average_return_pct': -0.4008, 'median_return_pct': -1.7407}, 'displaced': {'trades': 16, 'wins': 8, 'win_rate_pct': 50.0, 'average_return_pct': 3.0762, 'median_return_pct': 0.0486}}

 extended
stage_emerging_score_70_full_theme_new_top5_breakout {'common': 135, 'added': {'trades': 69, 'wins': 27, 'win_rate_pct': 39.1304, 'average_return_pct': 0.8012, 'median_return_pct': -1.4914}, 'displaced': {'trades': 21, 'wins': 9, 'win_rate_pct': 42.8571, 'average_return_pct': 3.148, 'median_return_pct': -1.7864}}
stage_emerging_score_70_full

## Takeaways

1. **主线跃迁是有覆盖的描述信号，但不是足够的买入质量信号。** 两个规则在四窗都有 33 至 54 条原始低分启动信号，却把账户胜率降到 50% 左右或更低。
2. **问题来自新增成交质量和账户机会成本。** 当前池候选新增/替换成交胜率约 28% 至 30%，中位收益为负；扩展池也只有约 38% 至 39%。这些交易还挤出了平均收益约 +2.2% 至 +3.1% 的原有成交。
3. **近期窗口不能推翻长窗反证。** 两候选 recent 收益约 +9.8% 至 +11.5%，但 train_a 都为负且四窗聚合显著落后，属于阶段性行情适配而非可推广优势。
4. **决策：拒绝 Round35 两候选。** 完整题材排名继续用于诊断和前向归因，不下调牛牛启动生产阈值；下一轮应优先研究‘启动后如何识别有效跟随’或持仓升级，而不是继续遍历入场分数。